# EEG → CLAP: Cross-Subject Auditory Attention Decoding (MAD-EEG)

**Struttura HDF5 reale:**
- `f[subj][stim_key]['response']` → `(20, T_eeg)` EEG 256 Hz
- `f[subj][stim_key]['soli']` → `(3, T_audio)` 44100 Hz — riga 0=attended, riga 1=unattended, riga 2=residuo
- `f[subj][stim_key]['stimulus']` → `(2, T_audio)` mix stereo

Il target (strumento attended) è l'**ultimo token** del nome stimolo: `..._Co` o `..._Fl`.

## 0. Imports & Setup

In [1]:
import os, glob, h5py, torch, librosa
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import LeaveOneGroupOut
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():      device = torch.device('cuda')
elif torch.backends.mps.is_available(): device = torch.device('mps')
else:                              device = torch.device('cpu')
print(f'Device: {device}')


Device: mps


## 1. Costanti globali

In [2]:
# ── Paths ───────────────────────────────────────────────────────────────
DATA_ROOT     = Path('MAD-MEG_dataset')  # cartella con madeeg_preprocessed.hdf5
HDF5_PATH     = DATA_ROOT / 'preprocessed/madeeg_preprocessed.hdf5'
EXCEL_PATH    = DATA_ROOT / 'behavioural_data.xlsx'
STIMULI_DIR   = DATA_ROOT / 'stimuli'
METADATA_CSV  = DATA_ROOT / 'metadata_built.csv'

# ── EEG ──────────────────────────────────────────────────────────────────
EEG_FS       = 256         # Hz — da documentazione MAD-EEG
N_CHANNELS   = 20
WINDOW_SEC   = 2.0         # durata finestra
HOP_SEC      = 1.0         # hop (50% overlap)
WINDOW_SAMP  = int(EEG_FS * WINDOW_SEC)   # 512
HOP_SAMP     = int(EEG_FS * HOP_SEC)      # 256

# ── Audio / CLAP ─────────────────────────────────────────────────────────
AUDIO_FS     = 44100       # Hz nel file HDF5
CLAP_FS      = 48000       # Hz richiesto da LAION-CLAP
CLAP_DIM     = 512

# ── soli layout (verificato dalla diagnostica) ────────────────────────────
# soli shape = (3, T_audio)
SOLI_IDX_ATTENDED   = 0    # riga 0: strumento attended
SOLI_IDX_UNATTENDED = 1    # riga 1: strumento unattended
# riga 2: residuo/mix — ignorata

# ── Training ─────────────────────────────────────────────────────────────
BATCH_SIZE   = 32
LR           = 1e-3
WEIGHT_DECAY = 1e-4
N_EPOCHS     = 30
TEMPERATURE  = 0.07

print('Costanti caricate.')
print(f'  WINDOW: {WINDOW_SAMP} samples ({WINDOW_SEC}s) | HOP: {HOP_SAMP} samples')


Costanti caricate.
  WINDOW: 512 samples (2.0s) | HOP: 256 samples


## 2. ClapAudioEncoder

Wrapper frozen di LAION-CLAP. Accetta waveform `(T,)` o `(C, T)` a qualsiasi sample rate
e restituisce un vettore L2-normalizzato di dimensione `CLAP_DIM`.

In [3]:
class ClapAudioEncoder:
    """
    Wrapper frozen LAION-CLAP.
    Se laion_clap non e' installato usa un encoder dummy (random) per
    testare il resto della pipeline senza dipendenze.
    """
    def __init__(self, device='cpu', enable_fusion=False):
        self.device     = device
        self.target_sr  = CLAP_FS
        print('Inizializzazione LAION-CLAP...')
        try:
            import laion_clap
            self.model = laion_clap.CLAP_Module(
                enable_fusion=enable_fusion,
                device=str(device)
            )
            self.model.load_ckpt()
            self.model.eval()
            self._available = True
            print('  CLAP caricato.')
        except Exception as e:
            print(f'  CLAP non disponibile ({e}) -> uso encoder dummy.')
            self._available = False

    @torch.no_grad()
    def encode(self, audio: np.ndarray, src_sr: int = AUDIO_FS) -> np.ndarray:
        """
        audio  : (C, T) o (T,)  float array
        src_sr : sample rate della sorgente
        return : (CLAP_DIM,) float32, L2-normalizzato
        """
        if audio.ndim > 1:
            audio = audio.mean(axis=0)          # stereo -> mono
        if src_sr != self.target_sr:
            audio = librosa.resample(
                audio.astype(np.float32),
                orig_sr=src_sr, target_sr=self.target_sr
            )
        audio = audio.astype(np.float32).reshape(1, -1)  # (1, T)

        if not self._available:
            v = np.random.randn(CLAP_DIM).astype(np.float32)
            return v / (np.linalg.norm(v) + 1e-8)

        emb = self.model.get_audio_embedding_from_data(x=audio, use_tensor=False)
        emb = emb.squeeze(0).astype(np.float32)
        return emb / (np.linalg.norm(emb) + 1e-8)


clap_encoder = ClapAudioEncoder(device=device)


Inizializzazione LAION-CLAP...


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 12385.37it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Load our best checkpoint in the paper.
The checkpoint is already downloaded
Load Checkpoint...
logit_scale_a 	 Loaded
logit_scale_t 	 Loaded
audio_branch.spectrogram_extractor.stft.conv_real.weight 	 Loaded
audio_branch.spectrogram_extractor.stft.conv_imag.weight 	 Loaded
audio_branch.logmel_extractor.melW 	 Loaded
audio_branch.bn0.weight 	 Loaded
audio_branch.bn0.bias 	 Loaded
audio_branch.patch_embed.proj.weight 	 Loaded
audio_branch.patch_embed.proj.bias 	 Loaded
audio_branch.patch_embed.norm.weight 	 Loaded
audio_branch.patch_embed.norm.bias 	 Loaded
audio_branch.layers.0.blocks.0.norm1.weight 	 Loaded
audio_branch.layers.0.blocks.0.norm1.bias 	 Loaded
audio_branch.layers.0.blocks.0.attn.relative_position_bias_table 	 Loaded
audio_branch.layers.0.blocks.0.attn.qkv.weight 	 Loaded
audio_branch.layers.0.blocks.0.attn.qkv.bias 	 Loaded
audio_branch.layers.0.blocks.0.attn.proj.weight 	 Loaded
audio_branch.layers.0.blocks.0.attn.proj.bias 	 Loaded
audio_branch.layers.0.blocks.0.norm2.we

## 3. Metadata builder

Costruisce un CSV trial-level dall'HDF5. Ogni riga = un trial (soggetto x stimolo).

**Naming convention stimolo:** `genre_song_ensemble_instruments_theme_spatial_target`
- `target` = ultimo token = strumento attended (es. `Co`, `Fl`)
- `soli[0]` = attended, `soli[1]` = unattended (verificato dalla diagnostica)

In [4]:
def build_metadata(hdf5_path: Path) -> pd.DataFrame:
    rows = []
    with h5py.File(hdf5_path, 'r') as f:
        for subj in f.keys():
            for stim_key in f[subj].keys():
                node = f[subj][stim_key]

                # Verifica che esistano response e soli
                if 'response' not in node or 'soli' not in node:
                    continue

                soli = node['soli']
                # Soli e' un Dataset shape (3, T_audio)
                # Serve almeno 2 righe (attended + unattended)
                if soli.ndim < 2 or soli.shape[0] < 2:
                    continue

                # Target = ultimo token del nome stimolo
                target = stim_key.split('_')[-1]  # es. 'Co' o 'Fl'

                rows.append({
                    'subject_id':   subj,
                    'stimulus_key': stim_key,
                    'target':       target,
                    'eeg_T':        node['response'].shape[1],
                    'audio_T':      soli.shape[1],
                })

    df = pd.DataFrame(rows)

    # Split LOSO: ultimi 2 soggetti = test, penultimo = val, resto = train
    subjects = sorted(df['subject_id'].unique())
    test_subj = set(subjects[-2:])
    val_subj  = {subjects[-3]}
    df['split'] = df['subject_id'].apply(
        lambda s: 'test' if s in test_subj else ('val' if s in val_subj else 'train')
    )
    return df


if METADATA_CSV.exists():
    metadata = pd.read_csv(METADATA_CSV)
    print(f'Caricato metadata: {len(metadata)} trial')
else:
    if not HDF5_PATH.exists():
        raise FileNotFoundError(f'HDF5 non trovato: {HDF5_PATH}')
    metadata = build_metadata(HDF5_PATH)
    metadata.to_csv(METADATA_CSV, index=False)
    print(f'Metadata costruito e salvato: {len(metadata)} trial')

print(metadata['split'].value_counts().to_string())
print(f'Soggetti: {sorted(metadata["subject_id"].unique())}')
metadata.head()


Metadata costruito e salvato: 246 trial
split
train    160
test      64
val       22
Soggetti: ['0001', '0002', '0003', '0004', '0005', '0007', '0008', '0009']


,subject_id,stimulus_key,target,eeg_T,audio_T,split
0,0001,classique_morceau1_duo_CoFl_theme1_stereo_Co,Co,7172,1235312,train
1,0001,classique_morceau1_duo_CoFl_theme1_stereo_Fl,Fl,7172,1235312,train
2,0001,classique_morceau1_duo_CoOb_theme1_stereo_Co,Co,7172,1235312,train
3,0001,classique_morceau1_duo_CoOb_theme2_stereo_Ob,Ob,6148,1058916,train
4,0001,classique_morceau1_duo_FlOb_theme1_stereo_Fl,Fl,7172,1235312,train


## 4. Dataset PyTorch

Legge dall'HDF5 a run-time (no pre-loading in RAM) usando `soli[0]` = attended, `soli[1]` = unattended.
Gli embedding CLAP vengono calcolati **on-the-fly** per finestra audio.

In [5]:
class EegClapWindowDataset(Dataset):
    """
    Sliding-window dataset su MAD-EEG.

    Struttura HDF5 attesa:
      f[subj][stim_key]['response'] -> (20, T_eeg)   float64
      f[subj][stim_key]['soli']    -> (3, T_audio)   float64
        soli[0] = attended stream
        soli[1] = unattended stream
        soli[2] = residuo (ignorato)

    __getitem__ ritorna:
      eeg_win  : Tensor (20, WINDOW_SAMP)  float32
      att_emb  : Tensor (CLAP_DIM,)        float32  L2-normalizzato
      unat_emb : Tensor (CLAP_DIM,)        float32  L2-normalizzato
    """

    def __init__(
        self,
        metadata:  pd.DataFrame,
        hdf5_path: Path,
        clap_enc:  ClapAudioEncoder,
        split:     str = 'train',
    ):
        assert split in {'train', 'val', 'test'}
        self.meta      = metadata[metadata['split'] == split].reset_index(drop=True)
        self.hdf5_path = hdf5_path
        self.clap_enc  = clap_enc
        self.index     = []   # lista di (subj, stim_key, win_start_eeg)
        self._build_index()

    def _build_index(self):
        """Pre-costruisce l'indice (subj, stim_key, win_start) senza caricare dati."""
        with h5py.File(self.hdf5_path, 'r') as f:
            for _, row in self.meta.iterrows():
                subj     = row['subject_id']
                stim_key = row['stimulus_key']
                T_eeg    = f[subj][stim_key]['response'].shape[1]

                starts = range(0, T_eeg - WINDOW_SAMP + 1, HOP_SAMP)
                for s in starts:
                    self.index.append((subj, stim_key, s))

        print(f'  [{self.meta["split"].iloc[0]}] {len(self.meta)} trial -> '
              f'{len(self.index)} finestre')

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        subj, stim_key, start = self.index[idx]
        end = start + WINDOW_SAMP
        audio_ratio = AUDIO_FS / EEG_FS
        a_start = int(start * audio_ratio)
        a_end   = int(end   * audio_ratio)

        with h5py.File(self.hdf5_path, 'r') as f:
            eeg_win  = f[subj][stim_key]['response'][:, start:end]          # (20, W)
            att_aud  = f[subj][stim_key]['soli'][SOLI_IDX_ATTENDED,  a_start:a_end]  # (T,)
            unat_aud = f[subj][stim_key]['soli'][SOLI_IDX_UNATTENDED, a_start:a_end] # (T,)

        eeg_win  = eeg_win.astype(np.float32)                # (20, W)
        att_emb  = self.clap_enc.encode(att_aud,  AUDIO_FS)  # (CLAP_DIM,)
        unat_emb = self.clap_enc.encode(unat_aud, AUDIO_FS)  # (CLAP_DIM,)

        return (
            torch.from_numpy(eeg_win),
            torch.from_numpy(att_emb),
            torch.from_numpy(unat_emb),
        )


train_ds = EegClapWindowDataset(metadata, HDF5_PATH, clap_encoder, split='train')
val_ds   = EegClapWindowDataset(metadata, HDF5_PATH, clap_encoder, split='val')
test_ds  = EegClapWindowDataset(metadata, HDF5_PATH, clap_encoder, split='test')

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_dl  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# Sanity check
eeg_ex, att_ex, unat_ex = train_ds[0]
print(f'EEG window shape:    {eeg_ex.shape}   (atteso: [20, {WINDOW_SAMP}])')
print(f'Att  embedding shape: {att_ex.shape}  (atteso: [{CLAP_DIM}])')
print(f'Unat embedding shape: {unat_ex.shape} (atteso: [{CLAP_DIM}])')


  [train] 160 trial -> 3533 finestre
  [val] 22 trial -> 522 finestre
  [test] 64 trial -> 1426 finestre
EEG window shape:    torch.Size([20, 512])   (atteso: [20, 512])
Att  embedding shape: torch.Size([512])  (atteso: [512])
Unat embedding shape: torch.Size([512]) (atteso: [512])


## 5. Modello CNN (EEGNet-inspired)

- **Conv temporale** `(1, 64)`: apprende filtri FIR per banda di frequenza
- **Depthwise spatial** `(C, 1)`: pesi anatomici per canale EEG
- **Projection head**: mappa in spazio CLAP + L2-normalizzazione

In [6]:
class EEGtoCLAP_CNN(nn.Module):
    def __init__(
        self,
        n_channels:  int   = N_CHANNELS,
        window_samp: int   = WINDOW_SAMP,
        clap_dim:    int   = CLAP_DIM,
        F1:          int   = 16,
        D:           int   = 2,
        dropout:     float = 0.25,
    ):
        super().__init__()
        F2 = F1 * D

        self.temporal = nn.Sequential(
            nn.Conv2d(1, F1, kernel_size=(1, 64), padding=(0, 32), bias=False),
            nn.BatchNorm2d(F1),
        )
        self.spatial = nn.Sequential(
            nn.Conv2d(F1, F2, kernel_size=(n_channels, 1), groups=F1, bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d((1, 4)),
            nn.Dropout(dropout),
        )
        flat_t = window_samp // 4
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(F2 * flat_t, 512),
            nn.ELU(),
            nn.Dropout(dropout * 1.6),
            nn.Linear(512, clap_dim),
        )

    def forward(self, x):         # x: (B, C, T)
        x = x.unsqueeze(1)        # (B, 1, C, T)
        x = self.temporal(x)      # (B, F1, C, T)
        x = self.spatial(x)       # (B, F2, 1, T//4)
        x = self.proj(x)          # (B, CLAP_DIM)
        return F.normalize(x, dim=-1)


model    = EEGtoCLAP_CNN().to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parametri trainabili: {n_params:,}')

# Test forward pass
with torch.no_grad():
    dummy = torch.randn(4, N_CHANNELS, WINDOW_SAMP).to(device)
    out   = model(dummy)
    print(f'Output shape: {out.shape}  (atteso: [4, {CLAP_DIM}])')
    print(f'L2 norm (deve essere ~1): {out.norm(dim=-1).mean():.4f}')


Parametri trainabili: 2,362,080
Output shape: torch.Size([4, 512])  (atteso: [4, 512])
L2 norm (deve essere ~1): 1.0000


## 6. InfoNCE Loss (simmetrica)

Spinge `EEG_pred` verso `CLAP_attended` e lontano da tutti gli altri embedding nel batch.
La versione simmetrica calcola la loss in entrambe le direzioni (EEG->audio e audio->EEG).

In [7]:
class SymmetricInfoNCE(nn.Module):
    def __init__(self, temperature: float = TEMPERATURE):
        super().__init__()
        self.tau = temperature

    def forward(self, eeg_emb: torch.Tensor, audio_emb: torch.Tensor) -> torch.Tensor:
        # eeg_emb, audio_emb: (B, D) gia' L2-normalizzati
        logits     = torch.matmul(eeg_emb, audio_emb.T) / self.tau  # (B, B)
        labels     = torch.arange(len(eeg_emb), device=eeg_emb.device)
        loss_e2a   = F.cross_entropy(logits,   labels)
        loss_a2e   = F.cross_entropy(logits.T, labels)
        return (loss_e2a + loss_a2e) / 2


criterion = SymmetricInfoNCE()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)
print('Loss, optimizer e scheduler pronti.')


Loss, optimizer e scheduler pronti.


## 7. Training loop

In [8]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for eeg, att_emb, _ in loader:
        eeg     = eeg.to(device)
        att_emb = att_emb.to(device)
        optimizer.zero_grad()
        pred    = model(eeg)
        loss    = criterion(pred, att_emb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(eeg)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def eval_accuracy(model, loader, device):
    """Binary AAD accuracy: corretto se sim(pred, att) > sim(pred, unatt)."""
    model.eval()
    correct, total = 0, 0
    for eeg, att_emb, unat_emb in loader:
        pred     = model(eeg.to(device))
        att_emb  = att_emb.to(device)
        unat_emb = unat_emb.to(device)
        sim_att  = (pred * att_emb ).sum(-1)
        sim_unat = (pred * unat_emb).sum(-1)
        correct += (sim_att > sim_unat).sum().item()
        total   += len(eeg)
    return correct / total if total > 0 else 0.0


best_val_acc = 0.0
history = {'train_loss': [], 'val_acc': []}

for epoch in range(1, N_EPOCHS + 1):
    train_loss = train_epoch(model, train_dl, optimizer, criterion, device)
    val_acc    = eval_accuracy(model, val_dl, device)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['val_acc'].append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pt')

    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{N_EPOCHS} | loss={train_loss:.4f} | val_acc={val_acc:.3f}')

print(f'\nBest val accuracy: {best_val_acc:.3f}')

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))
ax1.plot(history['train_loss']); ax1.set_title('Train Loss'); ax1.set_xlabel('Epoch')
ax2.plot(history['val_acc']);    ax2.set_title('Val Accuracy'); ax2.set_xlabel('Epoch')
ax2.axhline(0.5, color='r', linestyle='--', label='chance'); ax2.legend()
plt.tight_layout(); plt.show()


Epoch   1/30 | loss=3.6664 | val_acc=0.542


KeyboardInterrupt: 

## 8. Ridge Regression baseline (LOSO)

Lower bound: features EEG appiattite -> Ridge -> predice embedding CLAP attended.
Valutato con Leave-One-Subject-Out per rispettare il protocollo cross-subject.

In [ ]:
def collect_flat(ds):
    """Raccogli tutte le finestre come (X_flat, Y_att, Y_unat, subjects)."""
    X, Y_att, Y_unat, subjs = [], [], [], []
    subj_list = ds.meta['subject_id'].values
    win_count = {}  # conta finestre per soggetto

    for eeg_t, att_t, unat_t in tqdm(DataLoader(ds, batch_size=256), desc='collect'):
        X.append(eeg_t.numpy().reshape(len(eeg_t), -1))
        Y_att.append(att_t.numpy())
        Y_unat.append(unat_t.numpy())

    # Associa soggetto a ogni finestra tramite l'indice
    for subj, stim_key, _ in ds.index:
        subjs.append(subj)

    return (
        np.vstack(X),
        np.vstack(Y_att),
        np.vstack(Y_unat),
        np.array(subjs)
    )


# Unisci train + val + test per LOSO completo
all_datasets = EegClapWindowDataset.__new__(EegClapWindowDataset)
all_meta  = pd.concat([train_ds.meta, val_ds.meta, test_ds.meta]).reset_index(drop=True)
# Raccogliamo direttamente dai tre dataset separati
X_all, Y_att_all, Y_unat_all, subj_all = [], [], [], []

for ds in [train_ds, val_ds, test_ds]:
    for i, (s, sk, st) in enumerate(ds.index):
        eeg_t, att_t, unat_t = ds[i]
        X_all.append(eeg_t.numpy().flatten())
        Y_att_all.append(att_t.numpy())
        Y_unat_all.append(unat_t.numpy())
        subj_all.append(s)

X_all     = np.stack(X_all)
Y_att_all = np.stack(Y_att_all)
Y_unat_all= np.stack(Y_unat_all)
groups    = np.array(subj_all)

loso   = LeaveOneGroupOut()
alphas = np.logspace(-3, 4, 15)
accs_ridge = []

for train_idx, test_idx in loso.split(X_all, Y_att_all, groups=groups):
    ridge = RidgeCV(alphas=alphas)
    ridge.fit(X_all[train_idx], Y_att_all[train_idx])
    pred  = ridge.predict(X_all[test_idx])

    def cos_sim(a, b):
        return (a * b).sum(1) / (np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1) + 1e-8)

    acc = (cos_sim(pred, Y_att_all[test_idx]) >
           cos_sim(pred, Y_unat_all[test_idx])).mean()
    accs_ridge.append(acc)
    held_out_subj = groups[test_idx][0]
    print(f'  Held-out {held_out_subj}: {acc:.3f}')

print(f'\nRidge LOSO accuracy: {np.mean(accs_ridge):.3f} +/- {np.std(accs_ridge):.3f}')
print(f'Chance level: 0.500')


## 9. Valutazione finale su test set (CNN)

In [ ]:
model.load_state_dict(torch.load('best_model.pt', map_location=device))
test_acc = eval_accuracy(model, test_dl, device)
print(f'CNN  test accuracy: {test_acc:.3f}')
print(f'Ridge LOSO (media): {np.mean(accs_ridge):.3f}')
print(f'Chance level:        0.500')

# Per-subject breakdown sul test set
print('\nPer-subject (test):')
for subj in sorted(metadata[metadata['split']=='test']['subject_id'].unique()):
    idx = [i for i,(s,_,_) in enumerate(test_ds.index) if s == subj]
    sub_ds = torch.utils.data.Subset(test_ds, idx)
    sub_dl = DataLoader(sub_ds, batch_size=64)
    acc_s  = eval_accuracy(model, sub_dl, device)
    print(f'  {subj}: {acc_s:.3f}')
